In [1]:
!ls /dev/video*

/dev/video0  /dev/video1


In [ ]:
from pynq import Overlay, allocate, MMIO
import numpy as np
import cv2
import time

ol = Overlay("HDMI_Pipeline_wrapper.bit")

WIDTH = 640
HEIGHT = 480
STRIDE = WIDTH * 3

# Framebuffer
frame = allocate(shape=(HEIGHT, WIDTH, 3), dtype=np.uint8)
frame[:] = 0
frame.flush()

# VTC
vtc_mmio = MMIO(0x43C00000, 0x10000)
vtc_mmio.write(0x00, 0x1)

# VDMA
vdma_mmio = MMIO(0x43000000, 0x10000)

MM2S_VDMACR        = 0x00
MM2S_VDMASR        = 0x04
MM2S_VSIZE         = 0x50
MM2S_HSIZE         = 0x54
MM2S_FRMDLY_STRIDE = 0x58
MM2S_SA1           = 0x5C
MM2S_SA2           = 0x60
MM2S_SA3           = 0x64

# Reset VDMA
vdma_mmio.write(MM2S_VDMACR, 0x4)
for _ in range(1000):
    if (vdma_mmio.read(MM2S_VDMACR) & 0x4) == 0:
        break

addr = frame.physical_address
vdma_mmio.write(MM2S_SA1, addr)
vdma_mmio.write(MM2S_SA2, addr)
vdma_mmio.write(MM2S_SA3, addr)

vdma_mmio.write(MM2S_FRMDLY_STRIDE, STRIDE)
vdma_mmio.write(MM2S_HSIZE, STRIDE)
vdma_mmio.write(MM2S_VDMACR, 0x3)   # run + circular
vdma_mmio.write(MM2S_VSIZE, HEIGHT)

print("VDMA DMACR:", hex(vdma_mmio.read(MM2S_VDMACR)))
print("VDMA DMASR:", hex(vdma_mmio.read(MM2S_VDMASR)))

# Camera
cap = cv2.VideoCapture(0, cv2.CAP_V4L2)
print("Camera opened:", cap.isOpened())

time.sleep(1.0)

# Warm-up
for _ in range(10):
    cap.read()

try:
    while True:
        ret, frame_cam = cap.read()
        if not ret:
            print("Failed to read frame")
            break

        # Make sure size matches HDMI pipeline
        frame_cam = cv2.resize(frame_cam, (WIDTH, HEIGHT))

        # OpenCV gives BGR, pipeline expects RGB
        frame_rgb = cv2.cvtColor(frame_cam, cv2.COLOR_BGR2RGB)

        # Copy to DDR framebuffer
        frame[:] = frame_rgb
        frame.flush()

except KeyboardInterrupt:
    print("Stopped")

cap.release()

VDMA DMACR: 0x10003
VDMA DMASR: 0x10000
Camera opened: True
